# Prepare the individual-microphone recordings — CPU runtime

This reads the completed AuDroK intake and copies **58 point-microphone WAVs
(1.867 GB; about 2.94 file-hours)** unchanged to a new Drive folder. Three array
means per measurement are excluded. Select a **CPU runtime**.

Each source archive is checked against the frozen intake SHA, staged on local
SSD, and each selected member receives a complete CRC, finite-audio and native
format/frame check. No resampling, normalization or window generation occurs.
This preserves raw input for later study; it does not establish audible event
labels, independent sessions, negative exposure or model performance.

Check the paths and available additional Drive allocation below, then Run all.
Use one active runtime per preparation folder. Completed files resume after an
interruption; original archives remain untouched. Working copies need about 4 GB
of local SSD and are removed after their selected members finish.


In [ ]:
SOURCE_DRIVE_PATH = '/content/drive/MyDrive/drone-detection-v2/intake-v1/source-archives'
PROJECT_DRIVE_PATH = '/content/drive/MyDrive/drone-detection-v2/point-preparation-v1'
ADDITIONAL_DRIVE_BUDGET_GIB = 3.0  # Available new allocation you have checked.
REPO_URL = 'https://github.com/StoneDerek/drone-detection.git'
FIRST_REF = 'main'


In [ ]:
import os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
MOUNT = Path('/content/drive')
if not os.path.ismount(MOUNT):
    raise RuntimeError('Drive is not mounted')
SOURCES, PROJECT = Path(SOURCE_DRIVE_PATH).resolve(), Path(PROJECT_DRIVE_PATH).resolve()
if any(not path.is_relative_to(MOUNT.resolve()) or path == MOUNT.resolve() for path in [SOURCES, PROJECT]):
    raise ValueError('Choose project folders inside mounted Drive')
if not SOURCES.is_dir():
    raise ValueError('Source folder is missing; check SOURCE_DRIVE_PATH')
if PROJECT.is_relative_to(SOURCES) or SOURCES.is_relative_to(PROJECT):
    raise ValueError('Keep preparation output separate from original source storage')
PROJECT.mkdir(parents=True, exist_ok=True)


In [ ]:
"""Standard-library-only bootstrap, embedded verbatim in the Colab notebook."""

import hashlib
import json
import os
import re
import subprocess
import uuid
from pathlib import Path


def git_output(checkout, *arguments):
    return subprocess.check_output(['git', '-C', str(checkout), *arguments], text=True).strip()


def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def pin_checkout(project, checkout, repo_url, first_ref='main'):
    """Resolve once, then reuse that exact revision after runtime replacement.

    One active runtime per project. Refuse dirty or unrelated local checkouts.
    The saved lock hash is also checked; a pin is never silently refreshed.
    """
    project, checkout = Path(project), Path(checkout)
    project.mkdir(parents=True, exist_ok=True)
    pin_path = project / 'bootstrap.json'
    pin = json.loads(pin_path.read_text()) if pin_path.exists() else None
    if pin is not None and (pin.get('schema_version') != 1
                            or pin.get('repo_url') != repo_url
                            or not re.fullmatch(r'[0-9a-f]{40}', pin.get('revision', ''))
                            or not re.fullmatch(r'[0-9a-f]{64}', pin.get('runtime_lock_sha256', ''))):
        raise ValueError('Invalid/different project pin; choose a new project folder deliberately')
    if checkout.exists() and any(checkout.iterdir()):
        if not (checkout / '.git').exists():
            raise ValueError('Checkout folder contains unrelated files')
        if git_output(checkout, 'remote', 'get-url', 'origin') != repo_url:
            raise ValueError('Checkout has a different repository origin')
        if git_output(checkout, 'status', '--porcelain'):
            raise ValueError('Local checkout changed; preserve those changes before continuing')
    else:
        checkout.mkdir(parents=True, exist_ok=True)
        subprocess.run(['git', 'init', '--quiet', str(checkout)], check=True)
        subprocess.run(['git', '-C', str(checkout), 'remote', 'add', 'origin', repo_url], check=True)
    target = pin['revision'] if pin else first_ref
    subprocess.run(['git', '-C', str(checkout), 'fetch', '--quiet', '--depth', '1', 'origin', target], check=True)
    revision = git_output(checkout, 'rev-parse', 'FETCH_HEAD')
    if pin and revision != pin['revision']:
        raise ValueError('Fetched revision differs from saved project pin')
    subprocess.run(['git', '-C', str(checkout), 'checkout', '--quiet', '--detach', revision], check=True)
    candidate = {'schema_version': 1, 'repo_url': repo_url, 'revision': revision,
                 'runtime_lock_sha256': file_hash(checkout / 'notebooks/runtime/uv.lock')}
    if pin is not None and pin != candidate:
        raise ValueError('Pinned environment lock changed')
    if pin is None:
        payload = json.dumps(candidate, indent=2, sort_keys=True).encode()
        temporary = pin_path.with_name('bootstrap.partial-' + uuid.uuid4().hex)
        temporary.write_bytes(payload)
        if pin_path.exists() and json.loads(pin_path.read_text()) != candidate:
            raise ValueError('Another bootstrap wrote a different pin; use one active runtime')
        os.replace(temporary, pin_path)
        if pin_path.read_bytes() != payload:
            raise ValueError('Project pin read-back failed')
    return candidate


REPO = Path('/content/drone-point-preparation')
PIN = pin_checkout(PROJECT, REPO, REPO_URL, FIRST_REF)
print(json.dumps(PIN, indent=2))


In [ ]:
import subprocess
import sys

UV_DIR = Path('/content/drone-point-bootstrap')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade',
                '--target', str(UV_DIR), 'uv==0.13.5'], check=True)
UV_ENV = dict(os.environ, PYTHONPATH=str(UV_DIR), UV_CACHE_DIR='/content/drone-point-uv-cache',
              UV_PYTHON_INSTALL_DIR='/content/drone-point-python')
subprocess.run([sys.executable, '-m', 'uv', 'sync', '--project', str(REPO),
                '--frozen', '--python', '3.12'], env=UV_ENV, check=True)
PYTHON = REPO / '.venv/bin/python'
JOB_ENV = dict(os.environ, PYTHONPATH=str(REPO / 'src'))


In [ ]:
subprocess.run([str(PYTHON), str(REPO / 'scripts/prepare_point_cache.py'),
                '--sources', str(SOURCES), '--output', str(PROJECT / 'prepared'),
                '--cache', '/content/drone-point-working-sources',
                '--budget-gib', str(ADDITIONAL_DRIVE_BUDGET_GIB)],
               cwd=REPO, env=JOB_ENV, check=True)
summary = json.loads((PROJECT / 'prepared/summary.json').read_text())
assert summary['completed_point_files'] == 58 and summary['native_bytes_unchanged']
print(json.dumps(summary, indent=2))


Run the flush cell before a planned disconnect. After replacing the runtime,
rerun this same notebook: it reuses its saved code/environment pin and verifies
completed point files without staging their original archives again. Missing
files are reconstructed from the locked sources. Changed/corrupt files or run
identities stop execution for review.

The output contains the original native WAV bytes, per-file receipts, quality
measurements and an unassigned recording index. Publisher recording labels stay
separate from audible event intervals and reviewed negatives. The study still
needs additional independent campaigns and matched negative exposure.


In [ ]:
drive.flush_and_unmount()
print('Drive flushed and unmounted. Prepared files can be verified on a replacement runtime.')
